# 25 — Validação do texto completo rastreável

## TL;DR

Esta etapa comprova, em cenário controlado, que o pipeline prioriza trechos de **Resultados** e **Conclusão**, preserva tabelas extraídas e associa cada citação à seção, página e URL de origem. Quando a fonte não fornece página, o valor permanece ausente; o sistema não inventa paginação.

## Contexto e método

Montamos um artigo aberto simulado com seções e páginas conhecidas. Em seguida: (1) fragmentamos o texto sem misturar seções; (2) recuperamos os trechos pertinentes com BM25; (3) priorizamos Resultados/Conclusão; e (4) executamos uma resposta controlada no mesmo contrato estruturado usado pelo Gemini. Assim validamos a regra de software sem depender da disponibilidade de uma API externa.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from fatofake.chunking import ChunkingConfig, chunk_article_content
from fatofake.gemini_evidence import EvidenceDocument, EvidencePassage, GeminiEvidenceAnalyzer
from fatofake.pmc import ArticleContent, ContentSection
from fatofake.retrieval import Bm25Index

## Dados controlados

O resultado principal e uma tabela estão na página 6. Isso nos permite testar exatamente a mensagem esperada pela interface: `Esse trecho foi encontrado na seção Resultados, página 6.`

In [2]:
claim = 'The intervention reduces symptoms in adults.'
content = ArticleContent(
    pmid='12345678',
    pmcid=None,
    doi='10.1000/controlled',
    abstract='The intervention was evaluated in adults.',
    full_text='Controlled full text',
    sections=(
        ContentSection('Introduction', 'Symptoms are common and require effective interventions.', 2),
        ContentSection('Methods', 'Adults were randomly assigned to intervention or control groups.', 4),
        ContentSection('Resultados', 'The intervention reduced symptoms by 18 percent in adults. Table 2 — Symptoms — intervention 41 — control 50.', 6),
        ContentSection('Conclusion', 'The intervention was associated with fewer symptoms in this sample.', 7),
    ),
    access_level='OPEN_ACCESS_FULL_TEXT',
    pubmed_url='https://pubmed.ncbi.nlm.nih.gov/12345678/',
    pmc_url='https://publisher.example/article.pdf',
)
chunks = chunk_article_content(content, ChunkingConfig(max_words=40, overlap_words=5))
[(chunk.section, chunk.page_number, chunk.text) for chunk in chunks]

[('Introduction',
  2,
  'Symptoms are common and require effective interventions.'),
 ('Methods',
  4,
  'Adults were randomly assigned to intervention or control groups.'),
 ('Resultados',
  6,
  'The intervention reduced symptoms by 18 percent in adults. Table 2 — Symptoms — intervention 41 — control 50.'),
 ('Conclusion',
  7,
  'The intervention was associated with fewer symptoms in this sample.')]

## Resultados

In [3]:
ranked = list(Bm25Index(chunks).search(claim, top_k=8))
def section_priority(title):
    name = title.casefold()
    if 'result' in name: return 0
    if 'conclu' in name: return 1
    if 'discuss' in name: return 2
    return 3

ranked.sort(key=lambda item: (section_priority(item.chunk.section), item.rank))
selected = tuple(item.chunk for item in ranked[:4])
passages = tuple(EvidencePassage(
    passage_id=chunk.chunk_id, text=chunk.text, section=chunk.section,
    page_number=chunk.page_number, source_url=chunk.source_url,
    content_scope=content.access_level,
) for chunk in selected)
[(passage.section, passage.page_number, passage.content_scope) for passage in passages]

[('Resultados', 6, 'OPEN_ACCESS_FULL_TEXT'),
 ('Conclusion', 7, 'OPEN_ACCESS_FULL_TEXT'),
 ('Methods', 4, 'OPEN_ACCESS_FULL_TEXT'),
 ('Introduction', 2, 'OPEN_ACCESS_FULL_TEXT')]

In [4]:
results_passage = next(item for item in passages if item.section == 'Resultados')
def controlled_post(_url, _payload):
    assessment = {
        'pmid': '12345678', 'relation': 'SUPPORTS', 'confidence': 0.91,
        'rationale': 'O trecho de resultados descreve um efeito compatível.',
        'evidence_quote': 'The intervention reduced symptoms by 18 percent in adults.',
        'passage_id': results_passage.passage_id,
        'study_design': 'RANDOMIZED_CLINICAL_TRIAL',
    }
    return {'candidates': [{'content': {'parts': [{'text': json.dumps({'assessments': [assessment]})}]}}]}

document = EvidenceDocument(
    pmid=content.pmid, title='Controlled study', abstract=content.abstract or '',
    source_url=content.pubmed_url, passages=passages,
)
assessment = GeminiEvidenceAnalyzer('controlled-key', post_json=controlled_post).analyze(claim, [document])[0]
location = f'Esse trecho foi encontrado na seção {assessment.evidence_section}, página {assessment.evidence_page}.'
assert assessment.evidence_section == 'Resultados'
assert assessment.evidence_page == 6
assert assessment.content_scope == 'OPEN_ACCESS_FULL_TEXT'
assert 'Table 2' in results_passage.text
location

'Esse trecho foi encontrado na seção Resultados, página 6.'

## Conclusões e próximos limites

A validação confirma a cadeia `documento → seção/página → chunk → passagem escolhida → citação literal`. O texto de Resultados foi escolhido, a tabela permaneceu no trecho e a página 6 chegou à resposta final. Para XML do PMC e páginas HTML, a seção continua rastreável, mas a página deve aparecer como indisponível porque essas fontes não oferecem paginação física confiável.

Este notebook valida comportamento técnico, não precisão clínica. A etapa seguinte deve medir recuperação em um conjunto anotado (`Recall@K`, `Precision@K` e MRR) e testar PDFs reais com diferentes layouts de tabelas.